# 16 — API Contracts, Versioning, Pagination, and Evolution

Good APIs remain understandable and compatible as implementations evolve. Explore explicit schemas, pagination tokens, error responses, versioning rules, and backward-compatibility trade-offs using offline data.

*Advanced · 90–150 min*

## You will learn
- Specify API request/response contracts
- Handle pagination and bounded traversal
- Design stable structured error responses
- Plan compatible schema evolution

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. API contracts and validation

A well-defined response includes stable fields and documented types. New optional fields are generally easier to add than silently repurposing existing fields. A consumer must validate external input, even if the server claims a schema.

In [ ]:
def normalize_response(raw):
    if not isinstance(raw, dict) or not isinstance(raw.get("items"), list):
        raise ValueError("items list required")
    version = raw.get("version", 1)
    if version not in (1, 2):
        raise ValueError("unsupported version")
    return {"version": version, "items": list(raw["items"])}

print(normalize_response({"items": [1, 2], "version": 2}))

**Review:** Which response changes would break an older client that expects a number?

### 2. Error envelopes

An error contract should distinguish stable machine-readable codes from human-friendly descriptions. Avoid leaking internal stack traces and secrets through public responses.

In [ ]:
def error_response(code, message, request_id):
    return {
        "error": {
            "code": code,
            "message": message,
            "request_id": request_id,
        }
    }

response = error_response("INVALID_ARGUMENT", "page_size must be positive", "request-01")
print(response["error"]["code"])

**Review:** Why should clients branch on stable codes rather than literal English messages?

### 3. Pagination and loop safeguards

Cursor pagination can be preferable when records are inserted concurrently, but every traversal needs a termination rule and maximum page count. Treat cursor tokens as opaque rather than trying to decode their implementation.

In [ ]:
PAGES = {
    "start": {"items": ["A", "B"], "next": "two"},
    "two": {"items": ["C"], "next": None},
}
def collect(fetch, cursor="start", max_pages=10):
    visited, collected = set(), []
    while cursor is not None:
        if cursor in visited or len(visited) >= max_pages:
            raise RuntimeError("loop or page limit")
        visited.add(cursor)
        page = fetch(cursor)
        collected.extend(page["items"])
        cursor = page.get("next")
    return collected

print(collect(PAGES.__getitem__))

**Review:** What does cursor pagination protect against that a simple offset may not?

### 4. Versioned serializers

Maintain stable old output when introducing new optional fields. Use explicit transformations rather than relying on dumping an object's complete __dict__, which could unintentionally disclose internal data.

In [ ]:
def serialize_product(product, version=1):
    common = {"sku": product["sku"], "name": product["name"]}
    if version == 1:
        return common
    if version == 2:
        return {**common, "category": product.get("category")}
    raise ValueError("unsupported API version")

record = {"sku": "P1", "name": "Pen", "category": "stationery"}
print(serialize_product(record, 1), serialize_product(record, 2))

**Review:** When is it preferable to release a new API version rather than add an optional field?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/04_production_systems_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/16', {1: '536c39726b0244d5', 2: '09c4033456900cd5', 3: 'd24b524d51b722ef', 4: 'cf3cc8fecd6cb795'})

**Exercise 1.** A paginated catalog contract: items, the next cursor and a clear error for bad input.

In [ ]:
def list_products(cursor, page_size, catalog):
    if page_size < 1 or page_size > 100:
        return {"error": {"code": "invalid_page_size", "status": 400}}
    start = int(cursor or 0)
    items = catalog[start:start + page_size]
    next_cursor = str(start + page_size) if start + page_size < len(catalog) else ___
    return {"items": items, "next": next_cursor}

catalog = list("abcde")
check(1, [list_products(None, 2, catalog), list_products("4", 2, catalog), list_products(None, 0, catalog)])

**Exercise 2.** v2 adds an optional field; v1 output never changes.

In [ ]:
def serialize(user, version):
    body = {"id": user["id"], "name": user["name"]}
    if version == ___:
        body["email"] = user.get("email")
    return body

user = {"id": 1, "name": "Ana", "email": "a@x.io"}
check(2, [serialize(user, 1), serialize(user, 2)])

**Exercise 3.** A client must stop when a server repeats a cursor.

In [ ]:
def collect(fetch):
    items, seen, cursor = [], set(), None
    while True:
        page = fetch(cursor)
        items += page["items"]
        cursor = page["next"]
        if cursor is None:
            return items
        if cursor in ___:
            raise RuntimeError(f"repeated cursor {cursor}")
        seen.add(cursor)

pages = {None: {"items": [1], "next": "a"}, "a": {"items": [2], "next": "b"}, "b": {"items": [3], "next": "a"}}
try:
    collect(pages.get)
    outcome = "no error"
except RuntimeError as error:
    outcome = str(error)
check(3, outcome)

**Exercise 4.** Changing a field from string to number breaks old clients. Which change is safe?

In [ ]:
safe = {
    "add an optional field": ___,
    "change price from \"9.99\" to 9.99": ___,
    "remove a field clients read": False,
}
check(4, safe)

## Common mistakes

- Never serialize arbitrary internal objects into public API responses.
- Use bounded pagination and validate cursor policies.
- Versioning does not excuse missing input validation.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Adding an optional field to an API response is usually…

   - **a)** Backward compatible
   - **b)** Forbidden
   - **c)** A breaking change

**2.** Cursor pagination should protect against…

   - **a)** Small pages
   - **b)** JSON
   - **c)** Repeated cursors (infinite loops)

**3.** A stable error response includes…

   - **a)** A machine-readable code and a safe message
   - **b)** A stack trace
   - **c)** Database credentials

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/16', {1: '37f1b73c95ce', 2: '8d26c0292075', 3: '758e2ed85f7f'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [17 — Transactions, Constraints, and Exactly-Once Semantics](17_sqlite_transactions_integrity.ipynb)